In [ ]:
import kagglehub
import pandas as pd
import os
#import ast
# import matplotlib.pyplot as plt
#from jinja2 import Template
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, recall_score, precision_score, f1_score,roc_curve
from xgboost import XGBClassifier
from sklearn.inspection import permutation_importance
import plotly.graph_objects as go

In [ ]:
path = kagglehub.dataset_download(
    "blastchar/telco-customer-churn"
)

csv_path = os.path.join(
    path,
    "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df = pd.read_csv(csv_path)

Data Cleaning
Coverting TotalCharges to numeric and removing NaN values (only 4 present in set.)
Establish dummy variables for categorical columns and prepare x_raw by removing target variable and ID.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

categorical_cols = df.select_dtypes(include=['object','str']).columns
categorical_cols = categorical_cols.drop('customerID')
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

x_raw = df_encoded.drop('Churn_Yes', axis=1)
x_raw = x_raw.drop('customerID', axis=1)


Clustering
Data is scaled and PCA to allow for simple clustering visualization. PCA is NOT currently used in modeling to preserve model explainability to business stakeholders.

Uses kmeans cluster for determining cluster size, normalized churn rates, total revenue by cluster, and average tenure by cluster.

Currently, clustering group size was chosen visually. Future iterations will use elbow or silhouette to validate.

Analysis:
Cluster 2: Highest revenue customers come from roughly 20% of the total customers, accounting for around 50% of all revenue. Unsurprisingly, they also have the largest average tenure. Effectively our "core" customers

Cluster 0: Second highest revenue, but also second most likely to churn. Biggest opportunity here for bolstering retention

Cluster 3: Has the highest churn, but only accounts for about 9% of the revenue overall. So there may be some value in targeting them for retention, but since they have both low tenure and medium revenue, efforts are likely better spent elsewhere as they are perhaps the most volatile group.

Cluster 4: Low revenue, high retention.

Cluster 1: Low revenue, Low tenure, Low churn. Not a good candidate for business efforts.

In [ ]:
x_scaled = StandardScaler().fit_transform(x_raw)
pca = PCA(n_components = 2)
x_pca = pca.fit_transform(x_scaled)
y = df['Churn'].map({'No': 0, 'Yes': 1})
X = x_pca


kmeans = KMeans(n_clusters=5, init='k-means++', random_state=12)
y_kmeans = kmeans.fit_predict(X)

cluster_df = df.copy()
cluster_df['Cluster'] = y_kmeans

display(
        cluster_df.groupby('Cluster')
        .agg(Number_of_Customers=('Cluster','count'))
        .sort_values(['Number_of_Customers'], ascending=False)
        )

display(cluster_df.groupby('Cluster')['Churn']
        .value_counts(normalize=True).round(2).mul(100)
        .unstack()
        .sort_values(by='Yes', ascending=False)        
)

display(
    cluster_df.groupby('Cluster')
    .agg(Total_Revenue=('TotalCharges','sum'))
    .sort_values(['Total_Revenue'], ascending=False) 
    .style.format({'Total_Revenue': '${:,.2f}'})
        
        
    )

display(
    cluster_df.groupby('Cluster')
    .agg(Avg_Tenure=('tenure','mean'))
    .sort_values(['Avg_Tenure'], ascending=False)
         .style.format({'Avg_Tenure': '{:.1f} months'})
)

#plt.scatter(X[:, 0], X[:, 1], c=y_kmeans, s=50, cmap='viridis')
#plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], s=200, c='red', marker='X')
#plt.show()

Model Building Function

Accepts raw data (x_raw), target variable (y), a list of modeling methodologies (methods), and a simple boolean that determines whether or not to draw the ROC Curve (GenPlot).

Function returns a dictionary that includes model name, accuracy score, precision, recall, F1, and AUC.

In [ ]:
def build_models(x_raw,y, methods,genPlot):
    x_train, x_test, y_train, y_test = train_test_split(x_raw, y, test_size=0.2, random_state=123, stratify=y)

    results = []
    
    fig = go.Figure()

    for method in methods:
        model = method #RandomForestClassifier(random_state=92)
        model_name = type(model).__name__
    #print(x_train)
        model.fit(x_train, y_train)
        predictions = model.predict(x_test)
        # print(model_name)
        accuracy = accuracy_score(y_test, predictions)

        # print(f"Model Accuracy: {accuracy * 100:.2f}%")    
        # print(f"Classification Report:\n {classification_report(y_test, predictions)}")
    
        probs = model.predict_proba(x_test)[:,1]
    
        if isinstance(model, LogisticRegression):
            importance = pd.Series(
                model.coef_[0],
                index=x_train.columns
            ).sort_values()
            #display(importance)
        elif hasattr(model, "feature_importances_"):
            importance = pd.Series(
                model.feature_importances_,
                index=x_train.columns
            ).sort_values(ascending=False)
            #display(importance)
    
        result = permutation_importance(
            model,
            x_test,
            y_test,
            n_repeats=10,
            random_state=123
        )
    
        perm_importance = pd.Series(
            result.importances_mean,
            index=x_test.columns
        ).sort_values(ascending=False)
        #display(perm_importance)
        
        fpr, tpr, _ = roc_curve(y_test, probs)
        auc = roc_auc_score(y_test, probs)
        
        
        fig.add_trace(
            go.Scatter(
                x=fpr,
                y=tpr,
                mode='lines',
                name=f'{model_name} (AUC={auc:.3f})'
            )
        )
    
        results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions),
        "recall": recall_score(y_test, predictions),
        "f1": f1_score(y_test, predictions),
        "auc": auc
        })

    if genPlot:
        fig.add_trace(
        go.Scatter(
                x=[0, 1],
                y=[0, 1],
                mode='lines',
                name='Random Guess',
                line=dict(dash='dash')
            )
        )

        fig.update_layout(
            title='ROC Curve Comparison',
            xaxis_title='False Positive Rate',
            yaxis_title='True Positive Rate',
            width=900,
            height=700
        )

        fig.show()
    

    return results


Model Building and Results

Defines the list of modeling methodologies to try, passes to the build_models function, stores the result in a dataframe that is then formatted for output to be sorted by AUC score, highlighting the maximum value of each scoring metric to compare model types.

Currently, the winning model is logistic regression for both its high scoring on all metrics except recall (losing slightly to random forest) and because it is also the simplist model. 

In future iterations stability will be checked for models, evaluate if clustering adds value to the model, perform cross-validation.
May further explore hyperparameter tuning though dataset is small enough that there is likely not much to be gained and may be subject to overfitting.

In [ ]:
models = [
    LogisticRegression(
        max_iter=5000,
        l1_ratio=0),
    RandomForestClassifier(
        n_estimators=50,
        max_depth=10,
        min_samples_leaf=50,
        random_state=42),
    XGBClassifier(
        max_depth=4,
        learning_rate=0.05,
        n_estimators=500)
]
model_results = (
    pd.DataFrame(
        build_models(x_raw,y,models,True)
        ).sort_values("auc", ascending=False)
        .reset_index(drop=True)
    )
model_results.index += 1
model_results.index.name = "Rank"

display_results = (model_results
    .sort_values("auc", ascending=False)
    .style
    .format("{:.3f}", subset=["accuracy","precision","recall","f1","auc"])
    .highlight_max(
        subset=["accuracy","precision","recall","f1","auc"],
        axis=0,
        color='blue'
        )
)



display(display_results)
